# 第 11 章　机器学习基础与金融应用

::: {.callout-note}
## 本章要点

1. **预测 vs 因果**：ML 的目标是最小化预测误差，不是估计因果效应
2. **正则化线性模型**：Lasso / Ridge / Elastic Net，特征选择直觉
3. **树模型**：随机森林、XGBoost、LightGBM，超参数调优
4. **评估指标**：分类（AUC-ROC / PR 曲线）vs 回归（MAE / RMSE）
5. **SHAP 可解释性**：特征重要性的正确解读，金融风控应用
6. **综合案例**：上市公司亏损预测（二分类）
:::


## 环境准备


In [ ]:
# ── 第 11 章　机器学习基础　──────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                              precision_recall_curve, classification_report)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

OUTPUT = 'output'
os.makedirs(OUTPUT, exist_ok=True)
RNG = np.random.default_rng(42)
print('环境就绪 ✓')


---

## 1　演示数据：上市公司亏损预测

**任务**：用 $t$ 年的财务指标预测公司在 $t+1$ 年是否亏损（净利润 < 0）。
这是典型的**二分类问题**，样本不均衡（亏损公司约占 15%）。

我们模拟 10 个财务特征：规模、杠杆、ROA、营业收入增长率、
现金流、国企哑变量等，以及一些纯噪声特征（用于演示特征选择）。


In [ ]:
# ── 1.1  生成模拟财务数据 ──────────────────────────────────────────────
N_FIRMS = 3000
N_FEAT  = 15

X_raw = RNG.normal(0, 1, (N_FIRMS, N_FEAT))
feat_names = ['Size','Leverage','ROA','RevGrowth','CFO',
              'Tangibility','Age','SOE','Coverage','Mktcap',
              'Noise1','Noise2','Noise3','Noise4','Noise5']

# 真实亏损概率（由前 10 个特征决定）
true_coef = np.array([-0.3, 0.8, -2.5, -0.6, -1.0,
                       -0.2, -0.1, -0.4, -0.5, -0.2,
                        0, 0, 0, 0, 0])  # 后 5 个是纯噪声
log_odds = -1.5 + X_raw @ true_coef
prob_loss = 1 / (1 + np.exp(-log_odds))
Y = RNG.binomial(1, prob_loss)   # 1 = 亏损

df_ml = pd.DataFrame(X_raw, columns=feat_names)
df_ml['loss'] = Y

print(f'样本量：{N_FIRMS}  亏损率：{Y.mean():.3f}（约 {Y.mean()*100:.1f}%）')
print(f'特征数：{N_FEAT}（前 10 个真实相关，后 5 个纯噪声）')

# 时序划分（模拟 5 年面板，用 TimeSeriesSplit）
TRAIN_SIZE = int(N_FIRMS * 0.7)
X_train = df_ml[feat_names].values[:TRAIN_SIZE]
X_test  = df_ml[feat_names].values[TRAIN_SIZE:]
y_train = df_ml['loss'].values[:TRAIN_SIZE]
y_test  = df_ml['loss'].values[TRAIN_SIZE:]
print(f'训练集：{len(y_train)}  测试集：{len(y_test)}')


---

## 2　正则化线性模型

### 2.1　Lasso / Ridge / Elastic Net

| 模型 | 惩罚项 | 特点 |
|------|--------|------|
| **Ridge** | $\lambda \sum \beta_j^2$（L2）| 压缩系数但不置零；适合所有特征都相关的场景 |
| **Lasso** | $\lambda \sum |\beta_j|$（L1）| 自动将无关特征系数置零；内置特征选择 |
| **Elastic Net** | $\alpha \cdot$L1 + $(1-\alpha) \cdot$L2 | 两者结合；处理高相关特征组 |

对于本章的亏损预测任务（有 5 个纯噪声特征），
Lasso 应该能把这 5 个特征的系数压缩到接近 0。


In [ ]:
# ── 2.1  Lasso 路径：系数随正则化强度的变化 ─────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import lasso_path

scaler = StandardScaler()
X_sc   = scaler.fit_transform(df_ml[feat_names].values)

# Logistic Lasso（分类任务用 Logistic 回归 + L1）
alphas = np.logspace(-3, 1, 50)
coef_path = []
for alpha in alphas:
    m = LogisticRegression(penalty='l1', C=1/alpha, solver='liblinear',
                           max_iter=500, random_state=42)
    m.fit(X_sc[:TRAIN_SIZE], y_train)
    coef_path.append(m.coef_[0])
coef_path = np.array(coef_path)

fig, ax = plt.subplots(figsize=(10, 5))
colors_true  = plt.cm.Blues(np.linspace(0.4, 0.9, 10))
colors_noise = plt.cm.Reds(np.linspace(0.4, 0.9, 5))
for j in range(10):
    ax.plot(np.log10(alphas), coef_path[:,j], color=colors_true[j],
            lw=1.5, label=feat_names[j] if j < 3 else None)
for j in range(10, 15):
    ax.plot(np.log10(alphas), coef_path[:,j], color=colors_noise[j-10],
            lw=1.5, linestyle='--')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_xlabel('log10(α)：正则化强度 →', fontsize=10)
ax.set_ylabel('系数', fontsize=10)
ax.set_title('Lasso 路径：真实特征（蓝色）vs 噪声特征（红色虚线）', fontsize=11)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch11_lasso_path.png', dpi=150, bbox_inches='tight')
plt.show()


---

## 3　树模型：随机森林与 Gradient Boosting

### 3.1　三种树模型的比较

| 模型 | 策略 | 关键超参数 | 适用场景 |
|------|------|-----------|----------|
| **随机森林** | 并行 Bagging | `n_estimators`, `max_depth`, `max_features` | 稳健，不容易过拟合，适合初始基准 |
| **XGBoost** | 串行 Boosting + 正则化 | `learning_rate`, `n_estimators`, `max_depth` | 表格数据竞赛标准，调参空间大 |
| **LightGBM** | Leaf-wise Boosting | 同 XGBoost，加 `num_leaves` | 比 XGBoost 快 3–10 倍，大规模数据首选 |


In [ ]:
# ── 3.1  三种模型训练与测试集 AUC 对比 ──────────────────────────────
from sklearn.linear_model import LogisticRegression as LR

models = {
    'Logistic (L1)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LR(penalty='l1', C=0.1, solver='liblinear', random_state=42))]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, verbosity=0),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        num_leaves=31, subsample=0.8, random_state=42, verbose=-1),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    prob = m.predict_proba(X_test)[:,1]
    auc  = roc_auc_score(y_test, prob)
    ap   = average_precision_score(y_test, prob)
    results[name] = {'AUC-ROC': auc, 'AP': ap, 'prob': prob}
    print(f'{name:<20}: AUC={auc:.4f}  AP={ap:.4f}')


In [ ]:
# ── 3.2  ROC 曲线 + PR 曲线并排 ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = ['steelblue','darkorange','green','purple']

for (name, res), col in zip(results.items(), colors):
    # ROC
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    ax1.plot(fpr, tpr, color=col, lw=1.8,
             label=f"{name} (AUC={res['AUC-ROC']:.3f})")
    # PR
    prec, rec, _ = precision_recall_curve(y_test, res['prob'])
    ax2.plot(rec, prec, color=col, lw=1.8,
             label=f"{name} (AP={res['AP']:.3f})")

ax1.plot([0,1],[0,1],'gray',lw=0.8,linestyle='--')
ax1.set_xlabel('假阳性率（FPR）'); ax1.set_ylabel('真阳性率（TPR）')
ax1.set_title('ROC 曲线', fontsize=11); ax1.legend(fontsize=8)

base_rate = y_test.mean()
ax2.axhline(base_rate, color='gray', lw=0.8, linestyle='--',
            label=f'随机基准（{base_rate:.3f}）')
ax2.set_xlabel('召回率（Recall）'); ax2.set_ylabel('精确率（Precision）')
ax2.set_title('PR 曲线（不均衡数据更有意义）', fontsize=11); ax2.legend(fontsize=8)

plt.suptitle('亏损预测模型评估', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch11_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()


---

## 4　SHAP 可解释性

### 4.1　为什么需要可解释性

树模型的「特征重要性」（如 XGBoost 的 `feature_importances_`）
只告诉你「这个特征被模型用了多少次」，
不告诉你「这个特征对某个具体预测的贡献是正还是负」。

**SHAP（SHapley Additive exPlanations，Lundberg & Lee, 2017）**
基于博弈论的 Shapley 值，分解每个特征对每个预测的边际贡献，
同时保证一致性和全局解释性。

金融风控中的典型用途：
- 解释为什么某笔贷款被拒绝（监管合规要求）
- 识别哪些财务指标是预测公司违约的关键驱动因素


In [ ]:
# ── 4.1  SHAP 分析（基于 LightGBM 模型）────────────────────────────
try:
    import shap
    SHAP_OK = True
    print(f'shap {shap.__version__} ✓')
except ImportError:
    SHAP_OK = False
    print('shap 未安装（pip install shap）')

if SHAP_OK:
    lgbm_model = models['LightGBM']
    explainer   = shap.TreeExplainer(lgbm_model)
    shap_vals   = explainer.shap_values(X_test)

    # 处理多类输出（二分类时取正类的 shap 值）
    if isinstance(shap_vals, list):
        sv = shap_vals[1]
    else:
        sv = shap_vals

    # Summary plot
    plt.figure(figsize=(9, 6))
    shap.summary_plot(sv, X_test, feature_names=feat_names,
                      plot_type='dot', show=False)
    plt.title('SHAP Summary Plot：特征对亏损预测的影响', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT}/ch11_shap.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 输出全局平均 SHAP 值（绝对值）
    mean_shap = pd.Series(np.abs(sv).mean(axis=0), index=feat_names)
    print('\n全局特征重要性（平均|SHAP|）：')
    print(mean_shap.sort_values(ascending=False).round(4).to_string())
else:
    # 退化：用内置特征重要性代替
    imp = pd.Series(models['LightGBM'].feature_importances_, index=feat_names)
    print('LightGBM 内置特征重要性（shap 未安装）：')
    print(imp.sort_values(ascending=False).to_string())


---

## 5　章末练习

**练习 1（Lasso 特征选择）**
用 5 折 TimeSeriesSplit 交叉验证选择 Lasso 的最优正则化参数，
输出被选中的特征列表（系数非零），
验证 5 个噪声特征是否都被正确排除。

**练习 2（超参数调优）**
用 `GridSearchCV`（搭配 `TimeSeriesSplit` 作为 cv 参数）
调优 LightGBM 的 `max_depth` 和 `num_leaves`，
输出最优参数组合，比较调优前后的测试集 AUC。

**练习 3（不均衡处理）**
对亏损预测任务，尝试以下两种处理不均衡的方法：
① `class_weight='balanced'`；② SMOTE 过采样（`pip install imbalanced-learn`）。
比较两种方法在 PR 曲线下面积（AP）上的效果差异。

**练习 4（SHAP 解读）**
找出测试集中亏损概率最高的 5 家公司，
用 `shap.force_plot` 或 `shap.waterfall_plot` 展示每家公司的预测解释，
写 2–3 句话说明驱动因素。
